# Reproduce and justify the selected national access-pressure model

**Inherited contract:** the validated 6,067-practice national matrix, fourteen-feature specification and locked preprocessing controls.

**Purpose:** show why the fourteen-feature K-Means model with three profiles is the national benchmark, then reproduce its frozen partition under one configuration-controlled contract.

Four activity and change features receive `log1p`; all fourteen features receive robust median/IQR scaling. K-Means uses three clusters, 100 initialisations, a maximum of 500 iterations, seed 2026 and the Lloyd algorithm.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_PATH = ROOT / 'configs' / 'reference_apr2025_mar2026.json'
from gpap2.config import load_config
REFERENCE_CONFIG = load_config(CONFIG_PATH)
AUTHORITY_MANIFEST = REFERENCE_CONFIG.resolve(REFERENCE_CONFIG.authority_checksum_file)
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 6)


## Method contract

This stage inherits the validated national matrix. It exposes the ordered feature and transformation contract, fitted robust-scaling metadata, model controls, retained selection evidence and reproduced outputs. Values come from the [reference configuration](../configs/reference_apr2025_mar2026.json), [preprocessing implementation](../src/gpap2/preprocessing.py), [model implementation](../src/gpap2/models.py), [analysis implementation](../src/gpap2/analysis.py) and [reporting helpers](../src/gpap2/notebook_reporting.py).

In [2]:
from gpap2.io import read_contract_csv, validate_authority_file

selection_paths = {
    'model roles': ROOT / 'outputs' / 'tables' / 'model_role_register.csv',
    'robustness': ROOT / 'outputs' / 'tables' / 'robustness_summary.csv',
    'national quality': ROOT / 'outputs' / 'tables' / 'national_profile_quality.csv',
}
for path in selection_paths.values():
    validate_authority_file(path, AUTHORITY_MANIFEST)

model_roles = read_contract_csv(selection_paths['model roles'])
robustness = read_contract_csv(selection_paths['robustness'])
national_quality = read_contract_csv(selection_paths['national quality'])
model_roles

,analytical_object,cohort,final_role,decision
0,14-feature national OCS–GPAD K-Means model,"6,067 practices",Primary national hard-partition access-pressur...,Retain unchanged
1,12-feature OCS construct-validity model,"6,067 practices",National construct-validity comparator,Retain as sensitivity only
2,14-feature matched CBT inbound control,"3,020 practices",Matched-cohort benchmark for the inbound exper...,Retain as inbound benchmark
3,17-feature CBT inbound sensitivity,"3,020 practices",Informative inbound-activity sensitivity,Retain without replacing the matched control
4,14-feature outcome-complete matched control,"1,456 practices",Cohort control in the earlier 14–17–21 comparison,Retain as part of the completed outcome sequence
5,17-feature outcome-complete fit,"1,456 practices",Immediate baseline for the outcome-composition...,"Retain as comparator, not as a new primary model"
6,20-feature CBT ILR model,"1,456 practices",Preferred composition-aware CBT outcome sensit...,Use when interpreting independent outcome balance
7,21-feature raw CBT outcome model,"1,456 practices",Raw-share feature-representation comparator,"Retain, but do not prefer as the outcome parti..."
8,Gaussian mixture model,National modelling cohort,Probabilistic membership and uncertainty compa...,Retain as supporting uncertainty evidence
9,Agglomerative clustering,National modelling cohort,Hierarchical structural diagnostic,Retain as supporting method-comparison evidence


In [3]:
selection_evidence = robustness.loc[
    robustness['robustness_domain'].isin(
        ['Algorithmic alignment', 'Assignment uncertainty', 'Feature sensitivity']
    )
]
selection_evidence

,robustness_domain,cohort_n,result,interpretation,limitation
0,Algorithmic alignment,6067,1411 practices (23.3%) differed between aligne...,"Related structure, not identical membership.",Algorithm choice remains material for some pra...
1,Assignment uncertainty,6067,1966 practices (32.4%) carry interpretive caution,Uncertainty is retained and mapped.,Profile assignment is not equally certain for ...
2,Feature sensitivity,5677,Q1 14-vs-12 agreement=91.6%; ARI=0.761,Broadly related structure with non-trivial rea...,The 12-feature diagnostic was not promoted.


In [4]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, silhouette_score
from gpap2.config import load_config
from gpap2.io import read_contract_csv, sha256
from gpap2.models import align_labels, fit_primary_kmeans
from gpap2.notebook_reporting import (
    build_feature_contract_table,
    build_model_contract_table,
    build_output_contract_table,
    build_transformation_contract_table,
)
from gpap2.preprocessing import prepare_national_features

config = load_config(CONFIG_PATH)
source = config.resolve(config.input_directory) / config.specification('national_14').source_file
matrix = read_contract_csv(source)
reference_path = config.resolve(config.frozen_assignment_file)
reference = read_contract_csv(reference_path)
prepared = prepare_national_features(matrix, config)
model = fit_primary_kmeans(prepared.matrix, config.model)
final_profiles = align_labels(reference['kmeans_cluster'].to_numpy(), model.labels_)

summary = pd.DataFrame({
    'profile': np.sort(np.unique(final_profiles)),
    'practice_count': pd.Series(final_profiles).value_counts().sort_index().to_numpy(),
})
summary

,profile,practice_count
0,1,1753
1,2,2312
2,3,2002


In [5]:
feature_contract = build_feature_contract_table(config, 'national_14')
feature_contract

,feature_order,feature,transformation,included_in_model
0,1,ocs_submissions_per_1000_patient_months,log1p,True
1,2,ocs_clinical_share,unchanged,True
2,3,ocs_administrative_share,unchanged,True
3,4,gpad_appointments_per_1000_patient_months,log1p,True
4,5,gpad_dna_share,unchanged,True
5,6,gpad_face_to_face_share,unchanged,True
6,7,gpad_telephone_share,unchanged,True
7,8,gpad_same_day_share,unchanged,True
8,9,gpad_1_day_share,unchanged,True
9,10,gpad_2_to_7_days_share,unchanged,True


In [6]:
transformation_contract = build_transformation_contract_table(prepared)
transformation_contract

,feature_order,feature,transformation,fitted_median,pre_scaling_iqr,fitted_scale,iqr_gate_passed
0,1,ocs_submissions_per_1000_patient_months,log1p,4.090074,2.166030,2.166030,True
1,2,ocs_clinical_share,unchanged,0.666460,0.272465,0.272465,True
2,3,ocs_administrative_share,unchanged,0.286792,0.240173,0.240173,True
3,4,gpad_appointments_per_1000_patient_months,log1p,6.166856,0.360544,0.360544,True
4,5,gpad_dna_share,unchanged,0.040926,0.027324,0.027324,True
5,6,gpad_face_to_face_share,unchanged,0.661696,0.228128,0.228128,True
6,7,gpad_telephone_share,unchanged,0.220877,0.187455,0.187455,True
7,8,gpad_same_day_share,unchanged,0.417319,0.160886,0.160886,True
8,9,gpad_1_day_share,unchanged,0.070129,0.038942,0.038942,True
9,10,gpad_2_to_7_days_share,unchanged,0.175322,0.081201,0.081201,True


In [7]:
model_contract = build_model_contract_table(config, prepared)
model_contract

,setting,value,runtime_source
0,algorithm,K-Means,src/gpap2/models.py
1,clusters (k),3,reference configuration
2,initialisation,k-means++,src/gpap2/models.py
3,n_init,100,reference configuration
4,max_iter,500,reference configuration
5,random_state,2026,reference configuration
6,implementation,lloyd,reference configuration
7,centering,median,reference configuration
8,scaling,interquartile range (IQR),reference configuration
9,label alignment,maximum-agreement Hungarian assignment,src/gpap2/models.py


In [8]:
exact_agreement = float(np.mean(final_profiles == reference['kmeans_cluster'].to_numpy()))
ari = float(adjusted_rand_score(reference['kmeans_cluster'], final_profiles))
silhouette = float(silhouette_score(prepared.matrix, final_profiles))
assignment_sha256 = sha256(reference_path)
run_manifest = {
    'source_sha256': sha256(source),
    'frozen_assignment_sha256': sha256(reference_path),
    'rows': len(matrix),
    'features': list(prepared.feature_names),
    'exact_agreement': exact_agreement,
    'adjusted_rand_index': ari,
    'silhouette': silhouette,
    'random_seed': config.model.random_seed,
    'n_init': config.model.n_init,
    'max_iter': config.model.max_iter,
    'algorithm': config.model.algorithm,
}
runtime_manifest = ROOT / 'work' / 'notebooks' / 'national_profile_run_manifest.json'
runtime_manifest.parent.mkdir(parents=True, exist_ok=True)
runtime_manifest.write_text(json.dumps(run_manifest, indent=2) + '\n', encoding='utf-8')
output_contract = build_output_contract_table({
    'practices': len(matrix),
    'numeric features': len(prepared.feature_names),
    'profile sizes': ' | '.join(str(value) for value in summary['practice_count']),
    'silhouette': silhouette,
    'agreement with frozen assignment': exact_agreement,
    'ARI with frozen assignment': ari,
    'canonical aligned assignment SHA-256': assignment_sha256,
    'assignment output': reference_path.relative_to(ROOT).as_posix(),
    'runtime manifest': runtime_manifest.relative_to(ROOT).as_posix(),
})
output_contract

,measure,observed
0,practices,6067
1,numeric features,14
2,profile sizes,1753 | 2312 | 2002
3,silhouette,0.118431
4,agreement with frozen assignment,1.0
5,ARI with frozen assignment,1.0
6,canonical aligned assignment SHA-256,C8DA5EF5A270799FFDC5184D8A6ED127CA16982C5C4713...
7,assignment output,outputs/tables/national_profile_assignments.csv
8,runtime manifest,work/notebooks/national_profile_run_manifest.json


In [9]:
assert matrix[config.identifier].equals(reference[config.identifier])
assert exact_agreement == 1.0 and ari == 1.0
expected_sizes = national_quality.set_index('cluster')['practice_count'].to_dict()
assert summary.set_index('profile')['practice_count'].to_dict() == expected_sizes
assert transformation_contract['iqr_gate_passed'].all()
print('Every frozen national profile assignment was reproduced exactly.')

Every frozen national profile assignment was reproduced exactly.


## Decision

The configuration-controlled implementation exactly reproduces all 6,067 frozen assignments. The output uses final public profile labels 1 to 3; no zero-based label is presented as the analytical result.

**Stage handover:** The selected national profiles provide the reference partition against which restricted-cohort, representation, temporal, contextual, and geographic evidence is interpreted.